# WC_BADGE_D - ODI to Databricks Conversion

**Source File:** `WC_BADGE_D.txt`
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC
**Description:** This notebook processes badge dimension data, loading incremental updates from a source table into the `WC_BADGE_DETAILS_D` target dimension table.
It handles dropping and creating staging tables, loading data with incremental logic, and merging into the final dimension table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "12345", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

In [ ]:
-- MAGIC %sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT *
FROM v_etl_parameters"))

# Staging Table

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 30: Drop staging
DROP TABLE IF EXISTS workspace.gold.c_wc_badge_d;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 40: Create staging
CREATE TABLE workspace.gold.c_wc_badge_d
(
    INTEGRATION_ID      STRING,
    BADGE_ID            STRING,
    BADGE_STATUS        BIGINT,
    CONTACT_EMAIL       STRING,
    ORG_NAME            STRING,
    CREATED_DATE        TIMESTAMP,
    LAST_UPDATED_DATE   TIMESTAMP,
    ETL_PROC_WID        BIGINT
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 50: Load staging with incremental logic
INSERT INTO workspace.gold.c_wc_badge_d
SELECT
    b.INTEGRATION_ID,
    b.BADGE_ID,
    b.STATUS,
    b.CONTACT_EMAIL,
    b.ORGANISATION_NAME,
    b.CREATION_DATE,
    b.LAST_UPDATE_DATE,
    (SELECT etl_proc_wid FROM v_etl_parameters) AS etl_proc_wid
FROM workspace.src.src_badge_table b
WHERE b.LAST_UPDATE_DATE > to_date('2024-01-01', 'yyyy-MM-dd')
  AND b.LAST_UPDATE_DATE <= current_timestamp();

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS staging_record_count
FROM workspace.gold.c_wc_badge_d;

# Target Table Merge

In [ ]:
-- MAGIC %sql
-- Ensure target table exists before merge
CREATE TABLE IF NOT EXISTS workspace.gold.wc_badge_details_d
(
    INTEGRATION_ID      STRING,
    BADGE_ID            STRING,
    STATUS              BIGINT,
    CONTACT_EMAIL       STRING,
    ORG_NAME            STRING,
    CREATED_DATE        TIMESTAMP,
    LAST_UPDATED_DATE   TIMESTAMP,
    W_INSERT_DT         TIMESTAMP,
    W_UPDATE_DT         TIMESTAMP
) USING DELTA
PARTITIONED BY (CREATED_DATE)
COMMENT 'Dimension table for WC Badge Details';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO 100: Merge into target
MERGE INTO workspace.gold.wc_badge_details_d AS T
USING (
    SELECT
        INTEGRATION_ID,
        BADGE_ID,
        BADGE_STATUS        AS STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        CREATED_DATE,
        LAST_UPDATED_DATE
    FROM workspace.gold.c_wc_badge_d
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
WHEN MATCHED THEN
    UPDATE SET
        T.STATUS          = S.STATUS,
        T.CONTACT_EMAIL   = S.CONTACT_EMAIL,
        T.ORG_NAME        = S.ORG_NAME,
        T.LAST_UPDATED_DATE = S.LAST_UPDATED_DATE,
        T.W_UPDATE_DT     = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (
        BADGE_ID,
        STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        INTEGRATION_ID,
        CREATED_DATE,
        LAST_UPDATED_DATE,
        W_INSERT_DT,
        W_UPDATE_DT
    ) VALUES (
        S.BADGE_ID,
        S.STATUS,
        S.CONTACT_EMAIL,
        S.ORG_NAME,
        S.INTEGRATION_ID,
        S.CREATED_DATE,
        S.LAST_UPDATED_DATE,
        current_timestamp(),
        current_timestamp()
    );

# Optimize Target

In [ ]:
-- MAGIC %sql
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.gold.wc_badge_details_d
ZORDER BY (INTEGRATION_ID);

# Cleanup

In [ ]:
-- MAGIC %sql
-- Cleanup: Drop staging table
DROP TABLE IF EXISTS workspace.gold.c_wc_badge_d;

# Validation

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS final_target_record_count
FROM workspace.gold.wc_badge_details_d;

In [ ]:
-- MAGIC %sql
SELECT *
FROM workspace.gold.wc_badge_details_d
WHERE W_UPDATE_DT >= date_sub(current_date(), 1)
ORDER BY W_UPDATE_DT DESC
LIMIT 100;

# Conversion Notes & Manual Actions Required

1.  **Schema Inference:** Source schema `SRC_BADGE_TABLE` was inferred to `workspace.src` for `src_badge_table`. Target and staging tables (`C$_WC_BADGE_D`, `WC_BADGE_DETAILS_D`) were assigned to `workspace.gold`.
2.  **ETL Parameters:** `ETL_PROC_WID` was hardcoded to `12345` in the original SQL. It has been converted to a widget `${ETL_PROC_WID}` with `12345` as its default value. Additional standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ODI_SESS_NO`) have been added as widgets.
3.  **Date Filtering:** The `TO_DATE('2024-01-01', 'YYYY-MM-DD')` in the staging load is a hardcoded date. In a production scenario, this would typically be replaced by a dynamic `etl_last_extract_time` retrieved from a control table based on `${DATASOURCE_NUM_ID}`.
4.  **Target Table DDL:** The DDL for `WC_BADGE_DETAILS_D` was inferred from the `MERGE` statement's `INSERT` clause and staging table structure, including `W_INSERT_DT` and `W_UPDATE_DT` for audit purposes. `CREATED_DATE` was chosen for partitioning.
5.  **Data Type Mapping:** `VARCHAR2` -> `STRING`, `NUMBER` (without scale, e.g., BADGE_STATUS, ETL_PROC_WID) -> `BIGINT`, `DATE` -> `TIMESTAMP`. `TIMESTAMP` is used for `CREATED_DATE`, `LAST_UPDATED_DATE`, `W_INSERT_DT`, `W_UPDATE_DT` to accommodate time components.
6.  **Optimization:** An `OPTIMIZE ... ZORDER BY (INTEGRATION_ID)` statement has been added for the target table, preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` to ensure it runs effectively.